In [20]:
from dataclasses import asdict, dataclass
import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

In [5]:
def step_one_month(dt: datetime) -> datetime:
    """Increment datetime to the first day of the next month."""
    year = dt.year + (dt.month // 12)
    month = 1 if dt.month == 12 else dt.month + 1
    return dt.replace(year=year, month=month, day=1)

def get_monthly_return(annual_return, annual_std, size: int|tuple = 1):
    """Sample monthly returns based on the annually normally distributed return."""
    log_var_a = np.log(1 + (annual_std / annual_return)**2 )
    log_mu_a = np.log(annual_return) - 0.5 * log_var_a

    log_var_m = log_var_a / 12
    log_mu_m = log_mu_a / 12

    Xm = np.random.normal(loc=log_mu_m, scale=log_var_m**0.5, size=size)
    Rm = np.exp(Xm)
    if len(Rm) == 1:
        Rm = Rm[0].item()
    return Rm

In [39]:
@dataclass
class Record:
    value: int
    date: datetime.date
    event: str
    monthly_return: float|None = None
    tax: int|None = None


def records_to_df(records: list[Record]) -> pd.DataFrame:
    df = pd.DataFrame.from_records(asdict(r) for r in records)
    df["date"] = pd.to_datetime(df["date"])
    return df

In [62]:
class FondKonto:

    def __init__(self, start_date: datetime.datetime, start_capital: int, annual_return: float, annual_std: float):
        self.start_date = start_date if isinstance(start_date, datetime.date) else start_date.date()
        self.acquisition_cost = start_capital
        self.value = start_capital
        self.cost_basis = start_capital
        self.date = start_date
        self.annual_return = annual_return
        self.annual_std = annual_std
        self.records = []
        self._update_record(value=self.value, date=self.date, event="init")

    def __repr__(self):
        return f"<FondKonto value={self.value}, date={self.date}>"

    def step(self):
        r = get_monthly_return(
            annual_return=self.annual_return,
            annual_std=self.annual_std,
            size=1,
        )

        self.value = int(self.value * r)
        self.date = step_one_month(self.date)
        self._update_record(value=self.value, date=self.date, monthly_return=r, event="step")
        return self.value

    def sell(self, fraction: float = 1):
        value_to_sell = round(self.value * fraction)
        taxable_profit = fraction * max(0, self.value - self.cost_basis)
        tax = int(0.3 * taxable_profit)
        self.value = max(0, self.value - value_to_sell)
        self.cost_basis = round((1 - fraction) * self.cost_basis)
        self._update_record(value=self.value, date=self.date, tax=tax, event="sell")
        return value_to_sell - tax

    def _update_record(self, **kwargs):
        new_record = Record(**kwargs)
        self.records.append(new_record)

In [59]:
fk = FondKonto(
    start_date=datetime.date(2026, 6, 1),
    start_capital=700000,
    annual_return=1.2,
    annual_std=0.05,
)

print(fk)

for _ in range(12):
    fk.step()

print(fk)

sold_value = fk.sell(0.5)

print(fk)

fk.step()
fk.step()
fk.sell()

records_to_df(fk.records)

<FondKonto value=700000, date=2026-06-01>
<FondKonto value=812678, date=2027-06-01>
<FondKonto value=406339, date=2027-06-01>


,value,date,event,monthly_return,tax
0,700000,2026-06-01,init,NaN,NaN
1,694505,2026-07-01,step,0.992151,NaN
2,697524,2026-08-01,step,1.004348,NaN
3,703032,2026-09-01,step,1.007897,NaN
4,712913,2026-10-01,step,1.014055,NaN
5,717622,2026-11-01,step,1.006606,NaN
6,722434,2026-12-01,step,1.006706,NaN
7,732824,2027-01-01,step,1.014382,NaN
8,743463,2027-02-01,step,1.014518,NaN
9,766747,2027-03-01,step,1.031318,NaN


In [63]:
fk = FondKonto(
    start_date=datetime.date(2026, 6, 1),
    start_capital=100000,
    annual_return=0.9,
    annual_std=0.05,
)

for _ in range(10):
    fk.step()

fk.sell(0.2)

fk.annual_return = 1.25
for _ in range(10):
    fk.step()

fk.sell(0.2)
records_to_df(fk.records)

,value,date,event,monthly_return,tax
0,100000,2026-06-01,init,NaN,NaN
1,98242,2026-07-01,step,0.982428,NaN
2,92631,2026-08-01,step,0.942890,NaN
3,92128,2026-09-01,step,0.994572,NaN
4,93605,2026-10-01,step,1.016041,NaN
5,90468,2026-11-01,step,0.966491,NaN
6,87262,2026-12-01,step,0.964572,NaN
7,86480,2027-01-01,step,0.991043,NaN
8,87617,2027-02-01,step,1.013156,NaN
9,86849,2027-03-01,step,0.991235,NaN


In [85]:
class Isk:

    def __init__(
            self,
            start_date: datetime.datetime,
            start_capital: int,
            annual_return: float,
            annual_std: float,
            state_interest_rate: float = 0.0255,
        ):
        self.start_date = start_date if isinstance(start_date, datetime.date) else start_date.date()
        self.value = start_capital
        self.date = start_date
        self.current_year_deposits = 0
        self.taxable_value = 0
        self.annual_return = annual_return
        self.annual_std = annual_std
        self.state_interest_rate = state_interest_rate
        self.records = []
        self._update_record(value=self.value, date=self.date, event="init")

    def __repr__(self):
        return (
            f"<Isk value={self.value}, date={self.date}, "
            f"current_year_deposits={self.current_year_deposits}, "
            f"taxable_value={self.taxable_value}>"
        )

    def step(self):
        r = get_monthly_return(
            annual_return=self.annual_return,
            annual_std=self.annual_std,
            size=1,
        )

        self.value = int(self.value * r)
        self.date = step_one_month(self.date)
        self._update_record(value=self.value, date=self.date, monthly_return=r, event="step")
        self._update_tax()
        return self.value

    def deposit(self, value: int):
        self.value += value
        self.current_year_deposits += value
    
    def _update_tax(self):
        month = self.date.month
        if month in (4, 7, 10):
            self.taxable_value += self.value
        elif month == 1:
            self.taxable_value += self.value
            capital_profit =  (self.taxable_value + self.current_year_deposits) / 4
            capital_base = max(0, capital_profit - 300000)
            tax = round(0.3 * (0.01 + self.state_interest_rate) * capital_base)
            self._update_record(value=self.value, date=self.date, tax=tax, event="isk_tax")
            self.taxable_value = 0
            self.current_year_deposits = 0

    def _update_record(self, **kwargs):
        new_record = Record(**kwargs)
        self.records.append(new_record)

In [91]:
isk = Isk(
    start_date=datetime.date(2026, 1, 1),
    start_capital=552876,
    annual_return=1.2,
    annual_std=0.05,
)

for _ in range(12*10):
    isk.step()
    isk.deposit(2000)

df_rec = records_to_df(isk.records)
taxes_tot = df_rec["tax"].sum().astype(int).item()
isk.value - taxes_tot

3871884

In [104]:
fk = FondKonto(
    start_date=datetime.date(2026, 1, 1),
    start_capital=783158 + 259725,
    annual_return=1.15,
    annual_std=0.05,
)
fk.cost_basis -= 601872 + 120910

for _ in range(12*10):
    fk.step()
value = fk.value
fk.sell()
df_fk = records_to_df(fk.records)
taxes = df_fk["tax"].sum().item()
int(value - taxes)

2509088

In [ ]:
(
    df_fk.assign(cumtax=lambda x: x["tax"].fillna(0).cumsum().astype(int))
)

,value,date,event,monthly_return,tax,cumtax
0,1042883,2026-01-01,init,NaN,NaN,0
1,1037445,2026-02-01,step,0.994786,NaN,0
2,1049660,2026-03-01,step,1.011774,NaN,0
3,1051379,2026-04-01,step,1.001638,NaN,0
4,1066413,2026-05-01,step,1.014300,NaN,0
...,...,...,...,...,...,...
117,1808198,2035-10-01,step,1.014905,NaN,0
118,1779758,2035-11-01,step,0.984272,NaN,0
119,1780543,2035-12-01,step,1.000441,NaN,0
120,1807991,2036-01-01,step,1.015416,NaN,0
